# COS40007 - Portfolio Assessment 3
## Electrical grid stability: Random Forest, MLP and ANN classification

**Student:** Hoang Huu Hoan | **ID:** 105732188

**Engineering question:** How accurately do 11 operating parameters distinguish stable from
unstable simulated grid configurations, and how does the required ANN compare with Random
Forest and scikit-learn MLP?

This notebook contains all modelling and analysis code. Run all cells in order from this
directory using Python 3.12 and `requirements-lock.txt`. Paths are relative. The report and
`ASSISTANCE_LOG.md` disclose AI-generated implementation, execution, debugging and writing.
This is an assisted working analysis for student review; it is not represented as student-authored code.
Personal learning reflection and Canvas details are deferred in `SUBMISSION_TODO.md` at the user's request.

**Roadmap:** setup; data audit and fixed splits; five-fold traditional CV; ANN development;
frozen final refits; test comparison; persistence and independent reload; reflections and reproduction.
Results are added and verified at each actual development phase.

## 1. Setup and declared evaluation protocol

Use the Portfolio 2 stratified 70/30 split (seed 42): 7,000 development and 3,000 test rows.
This is a **previously inspected benchmark**: Task 1 explored all data and Task 2 examined
this test split. It is not untouched external validation. All new configuration and epoch
decisions use development evidence only. No test-based revisions are permitted.

Traditional CV uses five shuffled stratified folds (seed 42), with scaling inside each fold's
pipeline. Report sample fold SD (`ddof=1`), not confidence intervals. ANN trials use a fixed
5,600/1,400 stratified fit/validation split (seed 43), with scaling fitted on the 5,600 only.
Run one 10-epoch batch-32 pilot, then fresh batch-16/32/64 trials capped at 50 epochs, with
early stopping (patience 10, restore best weights) and LR reduction (factor 0.5, patience 5).
ANN-1 is batch 32 and ANN-2 is batch 64, regardless of the batch-16 development score.

Freeze each core ANN's best validation-loss epoch and all model settings before final test
inference. Refit all four models on identical scaled 7,000-row development inputs, with a
fresh common scaler. Final ANNs start afresh, use fixed initial Adam LR 0.001, and have no
validation callbacks; their schedules differ from trial schedules. Their saved histories and
`.h5` models refer to final refits. Required validation curves refer to development trials.
ANN CV is not performed; validation scores must not be labelled CV.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['OMP_NUM_THREADS'] = '2'
import sys, json, hashlib, pickle, warnings, time, copy, platform, subprocess, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import sklearn
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss)
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
import tensorflow as tf
from tensorflow import keras
from IPython.display import display, Markdown
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.experimental.enable_op_determinism()
threadpool_limits(limits=2)
SEED, VALIDATION_SEED = 42, 43
keras.utils.set_random_seed(SEED)
ROOT = Path.cwd()
assert (ROOT / 'data/Data_for_UCI_named.csv').is_file(), 'Run from portfolio_task3/'
for folder in ['models', 'results', 'figures', 'report']:
    (ROOT / folder).mkdir(exist_ok=True)
FEATURES = ['tau1','tau2','tau3','tau4','p2','p3','p4','g1','g2','g3','g4']
CLASS_MAP = {'stable': 0, 'unstable': 1}
CLASS_NAMES = ['stable', 'unstable']
REPOSITORY = 'https://github.com/henryhoang207/COS40007-Porfollio-Task'
VERSIONS = {p: importlib.metadata.version(p) for p in
            ['tensorflow','keras','scikit-learn','numpy','pandas','matplotlib','seaborn','h5py','nbformat','nbclient']}
VERSIONS['python'] = platform.python_version()
print(json.dumps(VERSIONS, indent=2))
print('CPU devices:', tf.config.list_physical_devices('CPU'))
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10})
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
def show_table(frame, filename=None):
    if filename:
        frame.to_csv(ROOT / 'results' / filename, index=False)
    display(frame)
def save_figure(name):
    plt.gcf().savefig(ROOT / 'figures' / name, bbox_inches='tight')
    plt.show()
    plt.close()
def explain(text):
    display(Markdown(text))
def build_ann(seed=SEED):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(len(FEATURES),), name='grid_features'),
        keras.layers.Dense(64, activation='relu', name='dense_64'),
        keras.layers.Dropout(0.2, name='dropout_1'),
        keras.layers.Dense(32, activation='relu', name='dense_32'),
        keras.layers.Dropout(0.2, name='dropout_2'),
        keras.layers.Dense(16, activation='relu', name='dense_16'),
        keras.layers.Dense(2, activation='softmax', name='class_probabilities'),
    ], name='grid_stability_ann')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model
smoke = build_ann()
smoke_x = np.zeros((3, 11), dtype=np.float32)
smoke_before = smoke(smoke_x, training=False).numpy()
import tempfile
with tempfile.TemporaryDirectory() as smoke_dir:
    smoke_path = Path(smoke_dir) / 'whole_model.h5'
    smoke.save(smoke_path)
    with h5py.File(smoke_path) as h:
        assert 'model_config' in h.attrs and 'model_weights' in h
    smoke_loaded = keras.models.load_model(smoke_path)
    np.testing.assert_allclose(smoke_before, smoke_loaded(smoke_x, training=False).numpy(), atol=1e-6, rtol=1e-6)
assert smoke.count_params() == 3410
smoke.summary()
save_json(ROOT / 'results/setup_verification.json', {
    'versions': VERSIONS, 'whole_model_hdf5_roundtrip': True,
    'ann_parameter_count': smoke.count_params(), 'cpu_training': True,
    'tensorflow_deterministic_ops': True, 'oneDNN_enabled': False,
})
print('PHASE 1 PASSED: imports, CPU, required architecture, whole-model HDF5 save/load.')

{
  "tensorflow": "2.20.0",
  "keras": "3.15.1",
  "scikit-learn": "1.7.2",
  "numpy": "2.2.6",
  "pandas": "2.3.3",
  "matplotlib": "3.10.7",
  "seaborn": "0.13.2",
  "h5py": "3.15.1",
  "nbformat": "5.10.4",
  "nbclient": "0.10.2",
  "python": "3.12.10"
}
CPU devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


Model: "grid_stability_ann"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_64 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probabilities (Dense)     │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,410 (13.32 KB)

 Trainable params: 3,410 (13.32 KB)

 Non-trainable params: 0 (0.00 B)

PHASE 1 PASSED: imports, CPU, required architecture, whole-model HDF5 save/load.
